In [4]:
import pandas as pd
from pathlib import Path

RES = Path("../results/fase6")   # sube de notebooks/ a tesis-cnn/ y entra a results/fase6
print(RES.resolve(), RES.exists())   # debe imprimir True

a = pd.read_csv(RES / "f1_por_participante_ce1.csv")
b = pd.read_csv(RES / "metricas_por_video.csv")

print(a.shape); print(a.head(20)); print(a.columns.tolist())
print(b.shape); print(b.head(6)); print(b.columns.tolist())

C:\Users\FIORELLA\PROYECTOS\tesis-cnn\results\fase6 True
(9, 4)
  participante        O1        O2  diferencia_O2_O1
0          P01  0.909091  0.952381          0.043290
1          P02  0.640000  0.666667          0.026667
2          P03  0.736842  0.763636          0.026794
3          P04  0.555556  0.153846         -0.401709
4          P05  0.615385  0.666667          0.051282
5          P06  0.709677  0.666667         -0.043011
6          P07  0.666667  0.666667          0.000000
7          P08  0.352941  0.400000          0.047059
8          P09  0.782609  0.631579         -0.151030
['participante', 'O1', 'O2', 'diferencia_O2_O1']
(36, 8)
  participante              video condicion enfoque  TP  FP  FN        F1
0          P01   V1_video1_normal    normal      O1   5   0   0  1.000000
1          P01   V1_video1_normal    normal      O2   5   0   0  1.000000
2          P01  V1_video2_pocaluz  poca_luz      O1   5   2   0  0.833333
3          P01  V1_video2_pocaluz  poca_luz      O2  

In [5]:
agg = b.groupby(["participante", "enfoque"])[["TP","FP","FN"]].sum().reset_index()
den = 2*agg.TP + agg.FP + agg.FN
agg["F1"] = 2*agg.TP / den
chk = agg.pivot(index="participante", columns="enfoque", values="F1")
print((chk.round(6) - a.set_index("participante")[["O1","O2"]].round(6)).abs().max())  # debe dar ~0

print(b[b.participante == "P04"])   # para entender el caso P04

enfoque
O1    0.0
O2    0.0
dtype: float64
   participante              video condicion enfoque  TP  FP  FN        F1
12          P04   V4_video1_normal    normal      O1   3   4   0  0.600000
13          P04   V4_video1_normal    normal      O2   0   4   3  0.000000
14          P04  V4_video2_pocaluz  poca_luz      O1   2   4   0  0.500000
15          P04  V4_video2_pocaluz  poca_luz      O2   1   3   1  0.333333


In [8]:
import pandas as pd
from pathlib import Path
RES = Path("../results/fase6")
b = pd.read_csv(RES / "benchmark_ce2_final_validado.csv")

# --- Verificaciones ---
print(b.shape)                                        # (54, 11)
print(b.groupby("video").repeticion.nunique().unique())   # debe dar [3]
print(b.participante.nunique(), b.video.nunique())        # 9 y 18
print(((b.frames * b.latencia_ms / 1000) - b.tiempo_s).abs().max())   # ~0
print((b.fps - 1000 / b.latencia_ms).abs().max())                      # ~0

# --- Tabla 49 ---
t49 = b[["fps", "latencia_ms"]].agg(["mean", "median", "std", "min", "max"]).round(2)
print(t49)

# --- Criterio CE2 (ambos a la vez, sobre el promedio) ---
print("FPS medio >= 15:", b.fps.mean() >= 15)
print("Latencia media <= 66.7:", b.latencia_ms.mean() <= 66.7)
print("Ejecuciones con FPS < 15:", (b.fps < 15).sum(), "| con latencia > 66.7:", (b.latencia_ms > 66.7).sum())

# --- Por condición (apoyo para el texto) ---
print(b.groupby("condicion")[["fps", "latencia_ms"]].mean().round(2))

(54, 11)
[3]
9 18
1.4210854715202004e-14
7.105427357601002e-15
          fps  latencia_ms
mean    21.65        46.37
median  21.52        46.48
std      1.42         2.97
min     18.65        39.08
max     25.59        53.62
FPS medio >= 15: True
Latencia media <= 66.7: True
Ejecuciones con FPS < 15: 0 | con latencia > 66.7: 0
             fps  latencia_ms
condicion                    
lentes     21.18        47.32
normal     22.09        45.45
poca_luz   21.23        47.28


In [10]:
m = pd.read_csv(RES / "metricas_por_video.csv")
print(m.condicion.value_counts())
print(sorted(m[m.condicion == "poca_luz"].participante.unique()))   # deben ser 6
print(sorted(m[m.condicion == "lentes"].participante.unique()))     # deben ser 3

condicion
normal      18
poca_luz    12
lentes       6
Name: count, dtype: int64
['P01', 'P04', 'P05', 'P06', 'P07', 'P09']
['P02', 'P03', 'P08']


In [9]:
import pandas as pd
from pathlib import Path
RES = Path("../results/fase6")

print(sorted(RES.glob("ce3*")))          # nombre completo del archivo

r = pd.read_csv(sorted(RES.glob("ce3*"))[0])
print(r.shape); print(r); print(r.columns.tolist())

[WindowsPath('../results/fase6/ce3_robustez_por_participante.csv')]
(9, 9)
  participante condicion_adversa  F1_normal_O1  F1_normal_O2  F1_adverso_O1  \
0          P01          poca_luz      1.000000      1.000000       0.833333   
1          P02            lentes      0.727273      0.800000       0.571429   
2          P03            lentes      0.774194      0.774194       0.692308   
3          P04          poca_luz      0.600000      0.000000       0.500000   
4          P05          poca_luz      0.571429      0.857143       0.666667   
5          P06          poca_luz      0.666667      0.666667       0.750000   
6          P07          poca_luz      0.857143      1.000000       0.545455   
7          P08            lentes      0.235294      0.333333       0.470588   
8          P09          poca_luz      0.727273      0.285714       0.833333   

   F1_adverso_O2  caida_F1_O1  caida_F1_O2  ventaja_robustez_O2  
0       0.909091     0.166667     0.090909             0.075758  
1 

In [11]:
import pandas as pd

df = pd.read_csv("../results/fase6/ce4_tiempos_alerta.csv")

print(df.shape)
print(df.columns.tolist())
print(df.to_string(index=False))

(166, 11)
['participante', 'video', 'condicion', 'enfoque', 'evento', 'frame_inicio_gt', 'frame_fin_gt', 'frame_alerta', 'detectado', 'retraso_frames', 'retraso_s']
participante             video condicion enfoque        evento  frame_inicio_gt  frame_fin_gt  frame_alerta  detectado  retraso_frames  retraso_s
         P01  V1_video1_normal    normal      O1 cierre_ocular              784           911         811.0          1            27.0   0.900385
         P01  V1_video1_normal    normal      O1 cierre_ocular             1139          1287        1159.0          1            20.0   0.666952
         P01  V1_video1_normal    normal      O1 cierre_ocular             1503          1683        1540.0          1            37.0   1.233860
         P01  V1_video1_normal    normal      O1       bostezo             1794          1964        1864.0          1            70.0   2.334330
         P01  V1_video1_normal    normal      O1       bostezo             2111          2267        2194